In [1]:
#!/usr/bin/env python3
"""
Ref: Raman, L., Van der Linden, M., Van der Eecken, K. et al. Shallow whole-genome sequencing of plasma cell-free DNA accurately differentiates small from non-small cell lung carcinoma. Genome Med 12, 35 (2020).
https://doi.org/10.1186/s13073-020-00735-4

CPA (copy-number profile abnormality) scores from ichorCNA 1 Mb outputs  -  APOLLO.

Run on the HPC (needs python3 + numpy + pandas):
    python3 compute_cpa.py
Only the two paths in the CONFIG block below ever need changing.

Per sample it reads  <ROOT>/<group>/<sample>/<sample>.cna.seg      (bin-level logR)
                     <ROOT>/<group>/<sample>/<sample>.seg.txt      (ichorCNA segments)
                     <ROOT>/<group>/<sample>/<sample>.params.txt   (tumor fraction, ploidy)

Two CPA versions are computed side by side (decide later which to use):
  A  "internal reference": healthy-donor reference is replaced by APOLLO samples with ichorCNA TF = 0
     (low-noise ones only). Samples that belong to the reference are scored with themselves left out.
  B  "reference-free": the healthy reference SD is replaced by the sample's own bin-to-bin noise (robust MAD).

Z-score per segment follows Raman et al. 2020, Eq. 1 (segment mean minus the mean of the reference samples' segment
means, divided by the SD of those segment means).  The final CPA sum is a RECONSTRUCTION from the TNBC paper's text
(sum of |Z| x segment length in 100-Mb units, divided by number of segments) -> see cpa_from_z(); compare it with the Zenodo code
(https://zenodo.org/records/16798720) and edit only that function if they differ.
"""
import glob, os
import numpy as np
import pandas as pd

np.random.seed(42)

# ----------------------------------------------------------------------------- CONFIG  (edit only these)
ROOT = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ichorCNA/1000kb"
OUT_DIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ichorCNA/CPA"
# ----------------------------------------------------------------------------- settings (change only if needed)
AUTOSOMES = {f"chr{i}" for i in range(1, 23)}   # X/Y excluded (PoN was autosome-normalised)
REF_TF_MAX = 0.0                  # reference = samples with ichorCNA tumor fraction <= this
REF_NOISE_QUANTILE = 0.75         # ...and noise (MAD) at or below this quantile of the TF = 0 samples
SD_FLOOR = 1e-3                   # floor for reference SD so a tiny SD cannot create an enormous z


# ----------------------------------------------------------------------------- reading
def load_sample(cna_seg_path):
    d = os.path.dirname(cna_seg_path)
    s = os.path.basename(cna_seg_path).replace(".cna.seg", "")
    bins = pd.read_csv(cna_seg_path, sep="\t")
    bins = bins.iloc[:, :10]
    bins.columns = ["chr", "start", "end", "cn", "event", "logR", "subclone", "cn_corr", "call_corr", "logR_cn"]
    bins = bins[bins.chr.isin(AUTOSOMES)].copy()
    bins["logR"] = pd.to_numeric(bins["logR"], errors="coerce")
    seg = pd.read_csv(os.path.join(d, f"{s}.seg.txt"), sep="\t")
    seg = seg[seg.chrom.isin(AUTOSOMES)].copy()
    par = pd.read_csv(os.path.join(d, f"{s}.params.txt"), sep="\t", nrows=1)
    tf, ploidy = float(par["Tumor Fraction"].iloc[0]), float(par["Ploidy"].iloc[0])
    return s, bins, seg, tf, ploidy


def segment_bin_index(bins, seg):
    """For each segment, the row positions (0-based) of the bins that lie inside it."""
    out = []
    for r in seg.itertuples(index=False):
        m = ((bins.chr.values == r.chrom) & (bins.start.values >= r.start) & (bins.end.values <= r.end))
        out.append(np.flatnonzero(m))
    return out


def noise_mad(logr, seg_idx):
    """Robust bin-to-bin noise: MAD of bin logR around its own segment mean (not inflated by real CNAs)."""
    res = []
    for idx in seg_idx:
        v = logr[idx]
        v = v[~np.isnan(v)]
        if v.size:
            res.append(v - v.mean())
    res = np.concatenate(res) if res else np.array([np.nan])
    return 1.4826 * np.nanmedian(np.abs(res - np.nanmedian(res)))


# ----------------------------------------------------------------------------- CPA
def cpa_from_z(z, length_mb):
    """RECONSTRUCTION (check against Zenodo). z: Z per segment, length_mb: segment length in Mb.
    cpa      = sum(|Z| x length[in 100-Mb units]) / number_of_segments
             = (mean |Z|) x (average segment length in 100 Mb). Healthy-like samples give ~1 (E|Z| ~ 0.8), and a noisy profile
               that is chopped into many small segments is NOT inflated, as the TNBC paper describes.
    mean_abs_z = length-weighted mean |Z| (no segment-count term), reported as a second, simpler score."""
    z, length_mb = np.asarray(z, float), np.asarray(length_mb, float)
    tot = np.nansum(np.abs(z) * length_mb)
    return tot / len(z) / 100.0, tot / length_mb.sum()


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    files = sorted(glob.glob(os.path.join(ROOT, "*", "*", "*.cna.seg")))
    print(f"found {len(files)} samples under {ROOT}")
    S = {}
    for f in files:
        s, bins, seg, tf, ploidy = load_sample(f)
        S[s] = dict(group=os.path.basename(os.path.dirname(os.path.dirname(f))), bins=bins, seg=seg, tf=tf, ploidy=ploidy)

    # common bin grid (all samples are expected to share the same 1 Mb bins)
    first = next(iter(S.values()))["bins"]
    key = (first.chr + ":" + first.start.astype(str)).values
    for s, v in S.items():
        k = (v["bins"].chr + ":" + v["bins"].start.astype(str)).values
        if len(k) != len(key) or not np.array_equal(k, key):
            v["bins"] = v["bins"].set_index(v["bins"].chr + ":" + v["bins"].start.astype(str)).reindex(key).reset_index(drop=True)
            first_bins = first
            v["bins"]["chr"] = first_bins.chr.values; v["bins"]["start"] = first_bins.start.values; v["bins"]["end"] = first_bins.end.values
    for s, v in S.items():
        v["logr"] = v["bins"]["logR"].values.astype(float)
        v["idx"] = segment_bin_index(v["bins"], v["seg"])
        v["noise"] = noise_mad(v["logr"], v["idx"])

    # ---- reference set for version A
    tf0 = [s for s, v in S.items() if v["tf"] <= REF_TF_MAX]
    cut = np.nanquantile([S[s]["noise"] for s in tf0], REF_NOISE_QUANTILE) if tf0 else np.nan
    ref = [s for s in tf0 if S[s]["noise"] <= cut]
    print(f"TF<={REF_TF_MAX} samples: {len(tf0)} | low-noise reference samples used: {len(ref)}")
    with open(os.path.join(OUT_DIR, "CPA_reference_samples.txt"), "w") as fh:
        fh.write("\n".join(ref) + "\n")
    B = np.column_stack([S[s]["logr"] for s in ref]) if ref else None     # bins x reference samples

    rows = []
    for s, v in S.items():
        seg, idx, r = v["seg"], v["idx"], v["logr"]
        keep = [i for i, ix in enumerate(idx) if ix.size and np.any(~np.isnan(r[ix]))]
        length_mb = ((seg["end"] - seg["start"] + 1) / 1e6).values[keep]
        seg_mean = np.array([np.nanmean(r[idx[i]]) for i in keep])

        # --- version A: internal reference (leave-one-out if the sample itself is in the reference)
        zA = np.full(len(keep), np.nan)
        if B is not None and len(ref) > 2:
            cols = [j for j, rs in enumerate(ref) if rs != s]
            for n, i in enumerate(keep):
                with np.errstate(all="ignore"):
                    m = np.nanmean(B[idx[i]][:, cols], axis=0)          # each reference sample's mean in this segment
                mu, sd = np.nanmean(m), max(np.nanstd(m, ddof=1), SD_FLOOR)
                zA[n] = (seg_mean[n] - mu) / sd
        # --- version B: reference-free (sample's own robust noise; reference mean assumed 0 after PoN normalisation)
        zB = seg_mean / max(v["noise"], SD_FLOOR)

        cA = cpa_from_z(zA, length_mb) if B is not None else (np.nan, np.nan)
        cB = cpa_from_z(zB, length_mb)
        calls = seg["Corrected_Call"].values
        slen = (seg["end"] - seg["start"] + 1).values.astype(float)
        rows.append(dict(
            sample=s, folder_group=v["group"], tumor_fraction=v["tf"], ploidy=v["ploidy"],
            n_segments=len(keep), n_bins_used=int(np.sum(~np.isnan(r))), bin_noise_mad=v["noise"],
            in_reference=s in ref,
            CPA_A_internal_ref=cA[0], meanAbsZ_A_internal_ref=cA[1],
            CPA_B_ref_free=cB[0], meanAbsZ_B_ref_free=cB[1],
            frac_genome_non_neutral=float(slen[calls != "NEUT"].sum() / slen.sum()),
            mean_abs_segment_logR=float(np.average(np.abs(seg_mean), weights=length_mb))))
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT_DIR, "APOLLO_CPA_scores.csv"), index=False)
    print("wrote", os.path.join(OUT_DIR, "APOLLO_CPA_scores.csv"), out.shape)
    print(out[["tumor_fraction", "CPA_A_internal_ref", "CPA_B_ref_free", "frac_genome_non_neutral"]]
          .corr(method="spearman").round(2))


if __name__ == "__main__":
    main()

/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


found 1027 samples under /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ichorCNA/1000kb
TF<=0.0 samples: 206 | low-noise reference samples used: 154
wrote /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ichorCNA/CPA/APOLLO_CPA_scores.csv (1026, 14)
                         tumor_fraction  CPA_A_internal_ref  CPA_B_ref_free  \
tumor_fraction                     1.00                0.90            0.91   
CPA_A_internal_ref                 0.90                1.00            0.97   
CPA_B_ref_free                     0.91                0.97            1.00   
frac_genome_non_neutral            0.76                0.78            0.78   

                         frac_genome_non_neutral  
tumor_fraction                              0.76  
CPA_A_internal_ref                          0.78  
CPA_B_ref_free                              0.78  
frac_genome_non_neutral                     1.00  
